# MLflow 3.13 GenAI Workshop

**Hands-on lab:** Prompt Registry · AI Gateway · Pydantic AI · Evaluation · Optimization

We build a small **support ticket triage** app end-to-end:

1. Store and version prompts in the **Prompt Registry**
2. Call models through the **AI Gateway** (no provider keys in app code)
3. Wire a **Pydantic AI** agent to the gateway + registry
4. **Evaluate** prompt quality on a labeled dataset
5. **Optimize** the prompt and promote the winner with aliases

> **Workshop tip:** Run cells top-to-bottom. After Part 1, open the MLflow UI → **Prompts** tab to inspect what you registered.

## 0. Prerequisites

Before starting, ensure you have:

1. **Python 3.12+** and project dependencies installed:
   ```bash
   uv sync
   ```
2. **Environment file** — copy and edit `Docker.env` in the project root:
   ```bash
   cp Docker.env Docker.env.local   # optional backup
   # edit MLFLOW_TRACKING_URI, MLFLOW_GATEWAY_ENDPOINT, MLFLOW_WORKSPACE, ...
   ```
   The notebook loads `Docker.env` automatically via `python-dotenv`.
3. **MLflow server + AI Gateway** running. Either:
   - Start from the example config:
     ```bash
     cp gateway_config.example.yaml gateway_config.yaml
     mlflow gateway start --config-path gateway_config.yaml --host 127.0.0.1 --port 5000
     ```
   - Or use an existing server (set `MLFLOW_GATEWAY_ENDPOINT` in `Docker.env`).

4. **Workspaces (optional today, ready for later):** set `MLFLOW_WORKSPACE=default` in `Docker.env`.
   When your server runs with `--enable-workspaces`, all registry / experiment / eval operations
   are scoped to that workspace. On servers without workspaces, the notebook detects this and
   runs in legacy mode automatically.

5. Open the MLflow UI — you'll use **Prompts**, **Experiments**, and **AI Gateway** tabs.

In [1]:
import difflib
import os
from pathlib import Path
from typing import Any, Literal

import mlflow
import requests
from loguru import logger
from mlflow.deployments import get_deploy_client
from mlflow.genai.optimize import MetaPromptOptimizer
from mlflow.genai.scorers import Guidelines, scorer
from openai import OpenAI
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel, OpenAIResponsesModelSettings
from pydantic_ai.providers.openai import OpenAIProvider


def find_docker_env(start: Path | None = None) -> Path:
    """Locate Docker.env from the notebook working directory or parents."""
    for directory in [start or Path.cwd(), *Path.cwd().parents]:
        candidate = directory / "Docker.env"
        if candidate.is_file():
            return candidate
    return Path.cwd() / "Docker.env"


def server_supports_workspaces(tracking_uri: str) -> bool:
    try:
        response = requests.get(
            f"{tracking_uri.rstrip('/')}/api/3.0/mlflow/server-info",
            timeout=5,
        )
        response.raise_for_status()
        return response.json().get("workspaces_enabled", False)
    except Exception as exc:
        logger.warning(f"Could not detect workspace support: {exc}")
        return False


def configure_mlflow_workspace(tracking_uri: str, workspace: str | None) -> str | None:
    """Activate a workspace when the server supports it."""
    if not workspace:
        os.environ.pop("MLFLOW_WORKSPACE", None)
        mlflow.set_workspace(None)
        return None
    if not server_supports_workspaces(tracking_uri):
        logger.info(
            f"Server has workspaces disabled; ignoring MLFLOW_WORKSPACE={workspace!r} for now"
        )
        os.environ.pop("MLFLOW_WORKSPACE", None)
        mlflow.set_workspace(None)
        return None
    os.environ["MLFLOW_WORKSPACE"] = workspace
    mlflow.set_workspace(workspace)
    return workspace


ENV_FILE = find_docker_env()
if not ENV_FILE.is_file():
    raise FileNotFoundError(
        f"Missing {ENV_FILE}. Copy Docker.env to the project root and edit your settings."
    )


In [2]:
import mlflow
import nest_asyncio
from dotenv import load_dotenv
from loguru import logger

load_dotenv(ENV_FILE, override=False)
nest_asyncio.apply()  # required for pydantic-ai in Jupyter

# --- Workshop configuration (from Docker.env) ---
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
GATEWAY_ENDPOINT = os.getenv("MLFLOW_GATEWAY_ENDPOINT", "my-ai-gateway-endpoint")
MLFLOW_WORKSPACE = os.getenv("MLFLOW_WORKSPACE", "default")
PROMPT_NAME = os.getenv("MLFLOW_PROMPT_NAME", "workshop-ticket-triage")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "mlflow-3-genai-workshop")
REASONING = {"effort": "medium"}  # Responses API thinking effort: low / medium / high
RUN_OPTIMIZATION = os.getenv("RUN_OPTIMIZATION", "true").lower() in {"1", "true", "yes"}

mlflow.set_tracking_uri(TRACKING_URI)
ACTIVE_WORKSPACE = configure_mlflow_workspace(TRACKING_URI, MLFLOW_WORKSPACE)
mlflow.set_experiment(EXPERIMENT_NAME)

gateway_client = get_deploy_client(TRACKING_URI)
# The OpenAI Responses API (/v1/responses) is served on the gateway's OpenAI
# passthrough route, not the unified /gateway/mlflow/v1 route (chat completions only).
GATEWAY_OPENAI_BASE_URL = f"{TRACKING_URI.rstrip('/')}/gateway/openai/v1"
openai_client = OpenAI(
    base_url=GATEWAY_OPENAI_BASE_URL,
    api_key="unused",  # provider auth is handled by the gateway
)

logger.info(f"Env file: {ENV_FILE}")
logger.info(f"Tracking URI: {TRACKING_URI}")
logger.info(f"Workspace: {ACTIVE_WORKSPACE or 'legacy (workspaces disabled)'}")
logger.info(f"Gateway endpoint: {GATEWAY_ENDPOINT}")
logger.info(f"Experiment: {EXPERIMENT_NAME}")

2026-10-03 18:49:15.635 | INFO     | __main__:configure_mlflow_workspace:48 - Server has workspaces disabled; ignoring MLFLOW_WORKSPACE='default' for now
2026-10-03 18:49:15.737 | INFO     | __main__:<module>:31 - Env file: /Users/setiabudi/Documents/PROJECTS/LEARNING/mlflow_3_guidelines/prompt_registry_and_ai_gateway_using_mlflow3/Docker.env
2026-10-03 18:49:15.738 | INFO     | __main__:<module>:32 - Tracking URI: http://localhost:5000
2026-10-03 18:49:15.738 | INFO     | __main__:<module>:33 - Workspace: legacy (workspaces disabled)
2026-10-03 18:49:15.738 | INFO     | __main__:<module>:34 - Gateway endpoint: my-ai-gateway-endpoint
2026-10-03 18:49:15.738 | INFO     | __main__:<module>:35 - Experiment: mlflow-3-genai-workshop


---
## Part 1 — Prompt Registry

The Prompt Registry treats prompts as **versioned artifacts** (like models in a model registry).

**Concepts:**
- `register_prompt()` — create a prompt or a new version (same name → auto-increment)
- `load_prompt("prompts:/name@alias")` — load by version or alias
- `.format(...)` — fill `{{ variables }}`
- `set_prompt_alias()` — promote/rollback without code changes

### 1.1 Register version 1 — a minimal triage prompt

In [3]:
V1_TEMPLATE = """Classify this support ticket into exactly one label:
PAYMENT, REFUND, RESCHEDULE, or OTHER.

Ticket: {{ ticket }}

Respond with the label only."""

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=V1_TEMPLATE,
    commit_message="Workshop v1: minimal classification prompt",
    tags={"workshop": "mlflow-3.13", "task": "ticket-triage"},
)

print(f"Registered {prompt_v1.name} v{prompt_v1.version}")
print(f"URI: {prompt_v1.uri}")

2026/10/03 18:49:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: workshop-dummy-prompt, version 4


Registered workshop-dummy-prompt v4
URI: prompts:/workshop-dummy-prompt/4


### 1.2 Load the prompt and fill variables — no hardcoded prompt strings in app code

In [4]:
loaded = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{prompt_v1.version}")

sample_ticket = "I was charged twice for my subscription this month."
filled = loaded.format(ticket=sample_ticket)

print("Formatted prompt preview:")
print(filled[:300], "...")

Formatted prompt preview:
Classify this support ticket into exactly one label:
PAYMENT, REFUND, RESCHEDULE, or OTHER.

Ticket: I was charged twice for my subscription this month.

Respond with the label only. ...


### 1.3 Register version 2 — add clearer rules (simulates a prompt improvement)

In [5]:
V2_TEMPLATE = """You are an expert support ticket triage assistant.

Task: Classify the ticket into exactly one of:
- PAYMENT — billing, duplicate charges, invoices
- REFUND — money back, cancellations, chargebacks
- RESCHEDULE — move/change appointment or delivery date
- OTHER — anything else

Rules:
- Output exactly one uppercase label
- No explanation, punctuation, or extra words

Ticket: {{ ticket }}"""

prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=V2_TEMPLATE,
    commit_message="Workshop v2: explicit label definitions and output rules",
)

print(f"Now at version {prompt_v2.version}")

2026/10/03 18:49:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: workshop-dummy-prompt, version 5


Now at version 5


### 1.4 Compare versions (like a Git diff)

In [6]:
v1 = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/1")
v2 = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/2")

print(f"v1: {v1.commit_message}")
print(f"v2: {v2.commit_message}\n")

diff = difflib.unified_diff(
    v1.template.splitlines(),
    v2.template.splitlines(),
    fromfile="v1",
    tofile="v2",
    lineterm="",
)
print("\n".join(diff))

v1: Workshop v1: minimal classification prompt
v2: Workshop v2: explicit label definitions and output rules

--- v1
+++ v2
@@ -1,6 +1,13 @@
-Classify this support ticket into exactly one label:
-PAYMENT, REFUND, RESCHEDULE, or OTHER.
+You are an expert support ticket triage assistant.
+
+Task: Classify the ticket into exactly one of:
+- PAYMENT — billing, duplicate charges, invoices
+- REFUND — money back, cancellations, chargebacks
+- RESCHEDULE — move/change appointment or delivery date
+- OTHER — anything else
+
+Rules:
+- Output exactly one uppercase label
+- No explanation, punctuation, or extra words
 
 Ticket: {{ ticket }}
-
-Respond with the label only.


### 1.5 Promote v2 to production via alias — apps load @production, not a version number

In [7]:
mlflow.genai.set_prompt_alias(
    PROMPT_NAME, alias="production", version=prompt_v2.version
)

production_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
print(f"Production alias → version {production_prompt.version}")

Production alias → version 5


**Check your work:** Open the MLflow UI → **Prompts** → `workshop-ticket-triage`. Compare v1 vs v2 and inspect the `production` alias.

## 🧪 Eksperimen Part 1 — dari slide *Prompt Registry*

Slide menjelaskan beberapa fitur yang belum dicoba di notebook asli: **chat prompt**, **model_config**, **search_prompts**, **URI `@latest`**, **cache TTL alias**, dan **rollback / hapus alias**. Kita coba satu per satu.

> Catatan: di server Anda nomor versi bisa saja bukan 1 dan 2 (prompt ini pernah diregistrasi berulang). Karena itu sel-sel di bawah memakai `.version` hasil `register_prompt`, bukan angka tetap.

### E1.1 Chat prompt (template berupa daftar pesan, slide "Register a prompt")

In [ ]:
CHAT_PROMPT_NAME = f"{PROMPT_NAME}-chat"

chat_prompt = mlflow.genai.register_prompt(
    name=CHAT_PROMPT_NAME,
    template=[
        {
            "role": "system",
            "content": (
                "You are a {{ tone }} support ticket triage assistant. "
                "Classify tickets into exactly one of PAYMENT, REFUND, RESCHEDULE, OTHER. "
                "Reply with the label only."
            ),
        },
        {"role": "user", "content": "Ticket: {{ ticket }}"},
    ],
    commit_message="Chat-style triage prompt (system + user)",
    tags={"workshop": "mlflow-3.13", "task": "ticket-triage", "format": "chat"},
)

messages = chat_prompt.format(tone="strict", ticket="Charged twice for one hotel night.")
print(f"Registered {chat_prompt.name} v{chat_prompt.version}")
print("format() pada chat prompt mengembalikan LIST pesan:")
for m in messages:
    print(" ", m)

### E1.2 `model_config` per versi prompt (slide "Load and use a prompt")

In [ ]:
V3_TEMPLATE = """You are an expert support ticket triage assistant.

Classify the ticket into exactly one label:
- PAYMENT — billing, duplicate charges, invoices
- REFUND — money back, cancellations, chargebacks
- RESCHEDULE — move/change appointment or delivery date
- OTHER — anything else (opening hours, general questions, feedback)

Examples:
Ticket: "My card was declined at checkout." -> PAYMENT
Ticket: "Please cancel and return my money." -> REFUND
Ticket: "Can we do Friday instead of Monday?" -> RESCHEDULE
Ticket: "Do you have a parking lot?" -> OTHER

Rules:
- Output exactly one uppercase label, nothing else.

Ticket: {{ ticket }}"""

prompt_v3 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=V3_TEMPLATE,
    commit_message="v3: few-shot examples + model_config",
    tags={"workshop": "mlflow-3.13", "task": "ticket-triage", "style": "few-shot"},
    model_config={"temperature": 0.0, "max_tokens": 16},
)
print(f"Registered v{prompt_v3.version}, model_config = {prompt_v3.model_config}")

### E1.3 Cari prompt dengan `search_prompts` (slide "Quick reference")

In [ ]:
try:
    found = mlflow.genai.search_prompts(filter_string="tags.task = 'ticket-triage'")
    print(f"Ditemukan {len(found)} prompt dengan tag task='ticket-triage':")
    for p in found:
        print(f"  - {p.name}")
except Exception as exc:
    print(f"search_prompts gagal ({exc.__class__.__name__}: {exc})")
    print("Sintaks filter berbeda antar versi; contoh dari slide: filter_string=\"task='summarization'\"")

### E1.4 Cara merujuk prompt: versi, alias, `@latest`, dan cache TTL

Slide: URI versi di-cache selamanya, URI alias punya TTL 60 detik, `cache_ttl_seconds=0` selalu mengambil data baru.

In [ ]:
by_version = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{prompt_v3.version}")
by_alias   = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
by_latest  = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
by_name    = mlflow.genai.load_prompt(PROMPT_NAME)   # tanpa URI = versi terbaru

print("by version :", by_version.version)
print("@production:", by_alias.version)
print("@latest    :", by_latest.version)
print("by name    :", by_name.version)

### E1.5 Rollback instan lewat alias, lalu kembalikan (slide "Aliases: promote & roll back")

In [ ]:
import time

# Simpan versi production saat ini supaya bisa dikembalikan di akhir
original_production = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production", cache_ttl_seconds=0
).version
print("Production sekarang  → v", original_production)

# 1) Promote v3
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=prompt_v3.version)
t0 = time.perf_counter()
print("Setelah promote v3   → v", mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production", cache_ttl_seconds=0).version)

# 2) Rollback tanpa mengubah kode aplikasi
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=original_production)
print("Setelah rollback     → v", mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production", cache_ttl_seconds=0).version)

# 3) Alias tambahan 'candidate' + hapus lagi
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="candidate", version=prompt_v3.version)
print("Alias candidate      → v", mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@candidate").version)
mlflow.genai.delete_prompt_alias(PROMPT_NAME, alias="candidate")
try:
    mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@candidate", cache_ttl_seconds=0)
except Exception as exc:
    print(f"Alias candidate sudah dihapus ({exc.__class__.__name__})")

### E1.6 Lineage: `load_prompt()` di dalam fungsi yang di-*trace*

Slide: memanggil `load_prompt()` di dalam fungsi ber-trace mencatat versi prompt sebagai dependensi. Setelah menjalankan sel ini, buka UI → **Experiments** → **Traces** dan lihat tab prompt pada trace.

In [ ]:
mlflow.openai.autolog()   # trace otomatis semua panggilan OpenAI SDK (lewat gateway)

@mlflow.trace
def traced_classify(ticket: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    resp = openai_client.responses.create(
        model=GATEWAY_ENDPOINT,
        reasoning=REASONING,
        input=prompt.format(ticket=ticket),
    )
    return resp.output_text.strip()

print(traced_classify("My invoice shows the wrong amount."))
print("Buka UI → Experiments → Traces untuk melihat prompt version yang terhubung.")

---
## Part 2 — AI Gateway

The AI Gateway is a single **OpenAI-compatible** entry point. Your app references an **endpoint name**; the gateway holds provider credentials and routes requests.

Two ways to call it:
- **OpenAI SDK (Responses API)** — point `base_url` at `{TRACKING_URI}/gateway/openai/v1` and call `client.responses.create(...)`
- **MLflow Deployments client** — `get_deploy_client(...).predict(...)`

> The Responses API is a passthrough route, so the endpoint must be backed by a provider that supports it (OpenAI, Azure/Databricks, LiteLLM). Anthropic-backed endpoints are not supported on this route.

### 2.1 Discover configured endpoints (may be empty if using UI-only setup)

In [8]:
try:
    endpoints = gateway_client.list_endpoints()
    print("Available endpoints:")  
    for ep in endpoints:
        name = ep["name"] if isinstance(ep, dict) else ep
        print(f"  - {name}")
except Exception as exc:
    print(f"Could not list endpoints ({exc.__class__.__name__}).")
    print(f"Using configured endpoint: {GATEWAY_ENDPOINT}")

Could not list endpoints (HTTPError).
Using configured endpoint: my-ai-gateway-endpoint


### 2.2 Call the gateway with the OpenAI Responses API — model = endpoint name

In [9]:
response = openai_client.responses.create(
    model=GATEWAY_ENDPOINT,
    reasoning=REASONING,
    input="In one sentence, what is the MLflow AI Gateway?",
)
print(response.output_text)

The MLflow AI Gateway is a centralized service that provides a unified interface for routing requests to and managing access to multiple large language model providers.


### 2.3 Alternative: MLflow Deployments client (when /endpoints API is enabled)


In [10]:
question = "In one sentence, what is the MLflow Prompt Registry?"
messages = [{"role": "user", "content": question}]

try:
    result = gateway_client.predict(
        endpoint=GATEWAY_ENDPOINT,
        inputs={"messages": messages},
    )
    print(result["choices"][0]["message"]["content"])
except Exception as exc:
    print(f"Deployments API unavailable ({exc.__class__.__name__}).")
    print("Falling back to OpenAI-compatible gateway path (same endpoint):")
    fallback = openai_client.responses.create(
        model=GATEWAY_ENDPOINT,
        reasoning=REASONING,
        input=messages,
    )
    print(fallback.output_text)

Deployments API unavailable (HTTPError).
Falling back to OpenAI-compatible gateway path (same endpoint):
The MLflow Prompt Registry is a centralized system for creating, versioning, organizing, and retrieving prompts used in generative AI applications.


In [11]:
# 2.4 Combine Prompt Registry + AI Gateway
def classify_ticket(ticket: str, prompt_uri: str | None = None) -> str:
    uri = prompt_uri or f"prompts:/{PROMPT_NAME}@production"
    prompt = mlflow.genai.load_prompt(uri)
    response = openai_client.responses.create(
        model=GATEWAY_ENDPOINT,
        reasoning=REASONING,
        input=prompt.format(ticket=ticket),
    )
    return response.output_text.strip()


for ticket in [
    "Charged twice for one hotel night.",
    "Need to move my appointment to next week.",
    "I want my money back for a cancelled flight.",
]:
    print(f"{ticket!r:55} → {classify_ticket(ticket)}")

'Charged twice for one hotel night.'                    → PAYMENT
'Need to move my appointment to next week.'             → RESCHEDULE
'I want my money back for a cancelled flight.'          → REFUND


**Check your work:** In the UI, open **AI Gateway** and confirm your endpoint. If usage tracking is enabled, you'll see traces after these calls.

## 🧪 Eksperimen Part 2 — dari slide *AI Gateway*

Yang dicoba: parameter `instructions`, parameter `temperature` di Deployments client, embeddings, kombinasi Registry + Deployments client, serta membangkitkan traffic untuk **usage tracking** dan melihat **traffic split**.

### E2.1 `instructions=` pada Responses API (slide "Option B")

In [ ]:
resp = openai_client.responses.create(
    model=GATEWAY_ENDPOINT,
    reasoning=REASONING,
    instructions="You are a concise assistant. Answer in at most 15 words.",
    input="What is MLflow AI Gateway?",
)
print(resp.output_text)

### E2.2 Deployments client dengan `temperature` + embeddings (slide "Option A")

In [ ]:
for temp in (0.0, 1.0):
    out = gateway_client.predict(
        endpoint=GATEWAY_ENDPOINT,
        inputs={
            "messages": [{"role": "user", "content": "Give me a one-line slogan for a support team."}],
            "temperature": temp,
        },
    )
    print(f"temperature={temp}: {out['choices'][0]['message']['content']}")

# Embeddings butuh endpoint bertipe llm/v1/embeddings — isi nama endpoint di Docker.env (opsional)
EMBEDDINGS_ENDPOINT = os.getenv("MLFLOW_EMBEDDINGS_ENDPOINT")
if EMBEDDINGS_ENDPOINT:
    emb = gateway_client.predict(endpoint=EMBEDDINGS_ENDPOINT, inputs={"input": ["hello", "world"]})
    print("Embedding dims:", len(emb["data"][0]["embedding"]))
else:
    print("MLFLOW_EMBEDDINGS_ENDPOINT belum di-set → bagian embeddings dilewati.")

### E2.3 Registry + Deployments client (slide "Combine with the Prompt Registry")

In [ ]:
prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
msgs = [{"role": "user", "content": prompt.format(ticket="Please reschedule my delivery to Friday.")}]
out = gateway_client.predict(endpoint=GATEWAY_ENDPOINT, inputs={"messages": msgs})
print(out["choices"][0]["message"]["content"])

### E2.4 Usage tracking & traffic split (slide "Traffic routing & fallbacks" dan "Usage tracking")

1. Di UI: **AI Gateway** → pilih endpoint → **Usage Tracking** → aktifkan → **Save changes**.
2. Jalankan sel di bawah untuk membangkitkan traffic, lalu lihat tab **Usage** (requests, latency p50/p90/p99, errors, token, cost).
3. Jika endpoint Anda memakai **traffic split** (mis. 50/50 dua model), sel ini juga menghitung model mana yang melayani tiap request.

> Pitfall dari slide: *fallback* hanya aktif jika total bobot traffic split tepat 100%. Fallback sendiri tidak bisa disimulasikan dari notebook — itu konfigurasi di endpoint.

In [ ]:
from collections import Counter

N_REQUESTS = 12
served_by = Counter()
latencies = []
for i in range(N_REQUESTS):
    t0 = time.perf_counter()
    r = openai_client.responses.create(
        model=GATEWAY_ENDPOINT,
        reasoning={"effort": "low"},
        input=f"Reply with the number {i}.",
    )
    latencies.append(time.perf_counter() - t0)
    served_by[getattr(r, "model", "unknown")] += 1

print("Model yang melayani request:", dict(served_by))
print(f"Latency rata-rata: {sum(latencies)/len(latencies):.2f}s | max: {max(latencies):.2f}s")

---
## Part 3 — Pydantic AI + Gateway + Prompt Registry

[Pydantic AI](https://ai.pydantic.dev/) builds typed agents. Because the MLflow gateway is OpenAI-compatible, point Pydantic AI's `OpenAIResponsesModel` at the same `base_url` and use the endpoint name as the model id.

### 3.1 Create a gateway-backed Pydantic AI model (Responses API)


In [12]:
pydantic_model = OpenAIResponsesModel(
    GATEWAY_ENDPOINT,
    provider=OpenAIProvider(
        base_url=GATEWAY_OPENAI_BASE_URL,
        api_key="unused",
    ),
    settings=OpenAIResponsesModelSettings(openai_reasoning_effort=REASONING["effort"]),
)

simple_agent = Agent(pydantic_model, system_prompt="You are a concise assistant.")
print(simple_agent.run_sync("Explain MLflow in one sentence.").output)

MLflow is an open-source platform for tracking machine-learning experiments, managing models, and deploying them.


### 3.2 Structured output — Literal types work best with gateway-backed models


In [13]:
class TriageResult(BaseModel):
    category: Literal["PAYMENT", "REFUND", "RESCHEDULE", "OTHER"]
    reason: str


def triage_with_agent(
    ticket: str, prompt_uri: str = f"prompts:/{PROMPT_NAME}@production"
) -> TriageResult:
    """Load governed prompt from registry, call gateway-backed agent, return typed result."""
    prompt = mlflow.genai.load_prompt(prompt_uri)
    agent = Agent(
        pydantic_model,
        output_type=TriageResult,
        system_prompt="Classify support tickets. Follow the user message instructions exactly.",
    )
    return agent.run_sync(prompt.format(ticket=ticket)).output


ticket = "Charged twice for one hotel night."
triage_result = triage_with_agent(ticket)
print(f"Ticket: {ticket}")
print(f"Category: {triage_result.category}")
print(f"Reason:   {triage_result.reason}")

Ticket: Charged twice for one hotel night.
Category: PAYMENT
Reason:   Duplicate charge for one hotel night


**Design note:** The registry owns the *prompt template*; the gateway owns *model access*; Pydantic AI adds *typed outputs* and agent tooling. Production apps typically load `prompts:/.../@production` so prompt updates don't require redeploys.

## 🧪 Eksperimen Part 3 — dari slide *Pydantic AI through the gateway*

Slide menunjukkan system prompt dari registry (`support-assistant`). Ada satu jebakan: `format()` pada **chat prompt** mengembalikan *list* pesan, bukan string, jadi kita ambil isi pesan `system`-nya dulu.

### E3.1 System prompt dari chat prompt registry

In [ ]:
def system_text_from_chat_prompt(prompt, **variables) -> str:
    msgs = prompt.format(**variables)
    return "\n".join(m["content"] for m in msgs if m["role"] == "system")

chat_prompt_loaded = mlflow.genai.load_prompt(f"prompts:/{CHAT_PROMPT_NAME}/{chat_prompt.version}")

def triage_with_registry_system_prompt(ticket: str, tone: str = "friendly") -> TriageResult:
    sys_text = system_text_from_chat_prompt(chat_prompt_loaded, tone=tone, ticket=ticket)
    agent = Agent(pydantic_model, output_type=TriageResult, system_prompt=sys_text)
    return agent.run_sync(f"Ticket: {ticket}").output

for tone in ("friendly", "strict"):
    r = triage_with_registry_system_prompt("I want my money back for a cancelled flight.", tone=tone)
    print(f"[{tone}] {r.category} — {r.reason}")

### E3.2 Anthropic-backed endpoint lewat `OpenAIChatModel` (opsional)

Slide: endpoint Anthropic tidak didukung di route Responses; pakai `OpenAIChatModel` dengan base URL `/gateway/mlflow/v1`. Set `MLFLOW_CHAT_ENDPOINT` di `Docker.env` untuk mencoba.

In [ ]:
CHAT_ENDPOINT = os.getenv("MLFLOW_CHAT_ENDPOINT")
if CHAT_ENDPOINT:
    from pydantic_ai.models.openai import OpenAIChatModel

    chat_model = OpenAIChatModel(
        CHAT_ENDPOINT,
        provider=OpenAIProvider(
            base_url=f"{TRACKING_URI.rstrip('/')}/gateway/mlflow/v1",
            api_key="unused",
        ),
    )
    chat_agent = Agent(chat_model, system_prompt="You are a concise assistant.")
    print(chat_agent.run_sync("Explain the MLflow AI Gateway in one sentence.").output)
else:
    print("MLFLOW_CHAT_ENDPOINT belum di-set → dilewati.")

---
## Part 4 — Prompt Evaluation

`mlflow.genai.evaluate()` runs your `predict_fn` over a dataset and scores outputs with custom or built-in scorers.

We'll compare behavior using:
- A **deterministic scorer** (exact label match)
- A **Guidelines scorer** (LLM-as-judge via the gateway)

In [14]:
EVAL_DATA = [
    {
        "inputs": {"ticket": "Charged twice for one hotel night."},
        "expectations": {"expected_response": "PAYMENT"},
    },
    {
        "inputs": {"ticket": "Need to move my appointment to next week."},
        "expectations": {"expected_response": "RESCHEDULE"},
    },
    {
        "inputs": {"ticket": "I want my money back for a cancelled flight."},
        "expectations": {"expected_response": "REFUND"},
    },
    {
        "inputs": {"ticket": "What are your opening hours on weekends?"},
        "expectations": {"expected_response": "OTHER"},
    },
]

print(f"Evaluation dataset: {len(EVAL_DATA)} examples")

Evaluation dataset: 4 examples


In [15]:
@scorer
def category_match(outputs: Any, expectations: dict[str, Any]) -> float:
    predicted = str(outputs).strip().upper().strip(".")
    expected = expectations["expected_response"]
    return float(predicted == expected)


def make_predict_fn(prompt_uri: str):
    def predict_fn(ticket: str) -> str:
        prompt = mlflow.genai.load_prompt(prompt_uri)
        response = openai_client.responses.create(
            model=GATEWAY_ENDPOINT,
            reasoning=REASONING,
            input=prompt.format(ticket=ticket),
        )
        return response.output_text.strip()

    return predict_fn


label_guidelines = Guidelines(
    name="label_only",
    guidelines="The response must be exactly one uppercase label (PAYMENT, REFUND, RESCHEDULE, or OTHER) with no extra words.",
    model=f"gateway:/{GATEWAY_ENDPOINT}",
)

In [16]:
# Evaluate v1 vs v2 side by side
results = {}
for version in [1, 2]:
    uri = f"prompts:/{PROMPT_NAME}/{version}"
    with mlflow.start_run(run_name=f"eval-v{version}") as run:
        eval_result = mlflow.genai.evaluate(
            data=EVAL_DATA,
            predict_fn=make_predict_fn(uri),
            scorers=[category_match, label_guidelines],
        )
        results[version] = eval_result
        print(f"v{version} metrics: {eval_result.metrics}")
        print(
            f"  → {TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}\n"
        )

2026/10/03 18:49:35 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/03 18:49:35 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?]

2026/10/03 18:49:46 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


v1 metrics: {'category_match/mean': np.float64(1.0), 'label_only/mean': np.float64(1.0)}
  → http://localhost:5000/#/experiments/4/runs/05a10b8bb08f46a58eda749cecbb90d4



Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?]

v2 metrics: {'category_match/mean': np.float64(1.0), 'label_only/mean': np.float64(1.0)}
  → http://localhost:5000/#/experiments/4/runs/a41ed79faadb4337a21e1ef82920cd22



**Check your work:** Open the evaluation runs in the MLflow UI. Compare `category_match/mean` and the per-row results table.

## 🧪 Eksperimen Part 4 — dari slide *Evaluate a prompt on a dataset*

Yang dicoba: dataset yang lebih besar dan lebih sulit, scorer bawaan `Correctness`, scorer kustom untuk **brevity**, dan A/B evaluasi **semua versi** (bukan hanya v1 vs v2) dengan promosi pemenang secara otomatis.

### E4.1 Dataset diperluas (12 contoh, termasuk kasus ambigu)

In [ ]:
def _ex(ticket, label):
    return {"inputs": {"ticket": ticket}, "expectations": {"expected_response": label}}

EVAL_DATA_EXT = EVAL_DATA + [
    _ex("My card was declined but money left my account.", "PAYMENT"),
    _ex("Please send me the invoice for last month.", "PAYMENT"),
    _ex("Cancel my order and return the money to my card.", "REFUND"),
    _ex("The chargeback I requested is still pending.", "REFUND"),
    _ex("Can we do Friday instead of Monday?", "RESCHEDULE"),
    _ex("Do you have a parking lot near the venue?", "OTHER"),
    _ex("Your app crashes when I open settings.", "OTHER"),
    _ex("I love the new design, great job!", "OTHER"),
]
print(f"EVAL_DATA_EXT: {len(EVAL_DATA_EXT)} contoh")

### E4.2 Scorer bawaan `Correctness` + scorer kustom brevity (slide "Custom scorers")

In [ ]:
from mlflow.genai.scorers import Correctness

@scorer
def brevity_scorer(outputs: Any) -> float:
    """1.0 jika jawaban sependek label; makin panjang makin kecil."""
    return min(1.0, 12 / max(len(str(outputs).strip()), 1))

correctness = Correctness(model=f"gateway:/{GATEWAY_ENDPOINT}")
print("Scorer siap: category_match, brevity_scorer, Correctness, label_guidelines")

### E4.3 A/B evaluasi semua versi, lalu promosikan pemenang

Pemenang dipilih dari `category_match/mean` (seri → pilih versi lebih baru). Alias `production` hanya diubah jika pemenang **tidak lebih buruk** dari production saat ini.

In [ ]:
all_versions = sorted(p.version for p in [prompt_v1, prompt_v2, prompt_v3])
ext_results = {}

for version in all_versions:
    uri = f"prompts:/{PROMPT_NAME}/{version}"
    with mlflow.start_run(run_name=f"eval-ext-v{version}") as run:
        res = mlflow.genai.evaluate(
            data=EVAL_DATA_EXT,
            predict_fn=make_predict_fn(uri),
            scorers=[category_match, brevity_scorer, correctness, label_guidelines],
        )
    ext_results[version] = res.metrics
    print(f"v{version}: {res.metrics}")

def _acc(v):
    return ext_results[v].get("category_match/mean", 0.0)

winner = max(all_versions, key=lambda v: (_acc(v), v))
current = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production", cache_ttl_seconds=0).version
print(f"\nPemenang: v{winner} (acc={_acc(winner):.2f}) | production sekarang: v{current}")

if winner != current and _acc(winner) >= _acc(current if current in ext_results else winner):
    mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=winner)
    print(f"→ production dipromosikan ke v{winner}")
else:
    print("→ production tidak diubah")

---
## Part 5 — Prompt Optimization

`mlflow.genai.optimize_prompts()` automatically rewrites a registered prompt to improve scorer metrics, saving the result as a **new registry version**.

For workshops we use **MetaPromptOptimizer** (fast, works with a handful of examples). For production with 100+ labeled rows, consider `GepaPromptOptimizer`.

In [17]:
if RUN_OPTIMIZATION:
    baseline = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/1")

    optimization_result = mlflow.genai.optimize_prompts(
        predict_fn=make_predict_fn(baseline.uri),
        train_data=EVAL_DATA,
        prompt_uris=[baseline.uri],
        optimizer=MetaPromptOptimizer(reflection_model=f"gateway:/{GATEWAY_ENDPOINT}"),
        scorers=[category_match],
    )

    optimized = optimization_result.optimized_prompts[0]
    print(f"Baseline score : {optimization_result.initial_eval_score}")
    print(f"Optimized score: {optimization_result.final_eval_score}")
    print(f"New prompt URI : {optimized.uri} (v{optimized.version})")
    print("\nOptimized template preview:")
    print(optimized.template[:500], "...")
else:
    print("Skipped optimization (RUN_OPTIMIZATION=False)")

2026/10/03 18:49:51 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/Users/setiabudi/Documents/PROJECTS/LEARNING/mlflow_3_guidelines/prompt_registry_and_ai_gateway_using_mlflow3/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/10/03 18:49:53 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: 4 training examples provided, using few-shot metaprompting
2026/10/03 18:49:53 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: Evaluating baseline prompts on training data...
2026/10/03 18:49:55 INFO mlflow.genai.optimize.optimizers.metaprompt_opt

🏃 View run colorful-perch-870 at: http://localhost:5000/#/experiments/4/runs/b112ed032ba941508db16a5713eb472c
🧪 View experiment at: http://localhost:5000/#/experiments/4
Baseline score : 1.0
Optimized score: 1.0
New prompt URI : prompts:/workshop-dummy-prompt/6 (v6)

Optimized template preview:
Classify the support ticket into exactly one category:
- PAYMENT: A charge, billing issue, or payment transaction, including duplicate charges.
- REFUND: A request to return money or receive reimbursement.
- RESCHEDULE: A request to change or move an appointment, booking, or scheduled service.
- OTHER: Any request that does not fit the categories above, such as a general information question.

Ticket: {{ ticket }}

Choose the category that best matches the customer's main request. If multiple to ...


Trace(trace_id=tr-90fb53bf3e5e21e6eca3e301f85b9a27)

In [18]:
if RUN_OPTIMIZATION:
    # Promote the optimized version — one line, no redeploy
    mlflow.genai.set_prompt_alias(
        PROMPT_NAME, alias="production", version=optimized.version
    )

    # Re-evaluate production alias on the same dataset
    with mlflow.start_run(run_name="eval-production-after-optimize") as run:
        final_eval = mlflow.genai.evaluate(
            data=EVAL_DATA,
            predict_fn=make_predict_fn(f"prompts:/{PROMPT_NAME}@production"),
            scorers=[category_match, label_guidelines],
        )
        print(f"Production metrics: {final_eval.metrics}")
        print(
            f"View run: {TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
        )

2026/10/03 18:50:00 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?]

Production metrics: {'category_match/mean': np.float64(1.0), 'label_only/mean': np.float64(1.0)}
View run: http://localhost:5000/#/experiments/4/runs/918e12cc716147e985ac6a59e2a597ec


## 🧪 Eksperimen Part 5 — dari slide *Choosing an optimizer* dan *Custom scorers*

Dua percobaan: (1) **objective berbobot** `0.7 × akurasi + 0.3 × brevity` lewat parameter `aggregation`, dan (2) **GEPA** vs **MetaPrompt** pada dataset yang sama. Hasil disimpan sebagai versi baru dengan alias *kandidat*, tidak langsung mengganti `production`.

> GEPA butuh paket tambahan: `pip install gepa`. Kedua sel ikut flag `RUN_OPTIMIZATION`.

### E5.1 Optimasi dengan objective berbobot (MetaPrompt)

In [ ]:
def weighted_objective(scores: dict[str, float]) -> float:
    return 0.7 * scores["category_match"] + 0.3 * scores["brevity_scorer"]

if RUN_OPTIMIZATION:
    base = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{prompt_v1.version}")
    res_weighted = mlflow.genai.optimize_prompts(
        predict_fn=make_predict_fn(base.uri),
        train_data=EVAL_DATA_EXT,
        prompt_uris=[base.uri],
        optimizer=MetaPromptOptimizer(reflection_model=f"gateway:/{GATEWAY_ENDPOINT}"),
        scorers=[category_match, brevity_scorer],
        aggregation=weighted_objective,
    )
    opt_weighted = res_weighted.optimized_prompts[0]
    mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="candidate-weighted", version=opt_weighted.version)
    print(f"Skor awal  : {res_weighted.initial_eval_score}")
    print(f"Skor akhir : {res_weighted.final_eval_score}")
    print(f"Versi baru : v{opt_weighted.version} (alias candidate-weighted)")
else:
    print("RUN_OPTIMIZATION=False → dilewati")

### E5.2 GEPA dengan anggaran kecil (slide "Optimize a prompt automatically")

In [ ]:
if RUN_OPTIMIZATION:
    from mlflow.genai.optimize import GepaPromptOptimizer

    base = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{prompt_v1.version}")
    res_gepa = mlflow.genai.optimize_prompts(
        predict_fn=make_predict_fn(base.uri),
        train_data=EVAL_DATA_EXT,
        prompt_uris=[base.uri],
        optimizer=GepaPromptOptimizer(
            reflection_model=f"gateway:/{GATEWAY_ENDPOINT}",
            max_metric_calls=60,   # slide memakai 300; dikecilkan agar cepat & hemat
        ),
        scorers=[category_match],
    )
    opt_gepa = res_gepa.optimized_prompts[0]
    mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="candidate-gepa", version=opt_gepa.version)
    print(f"GEPA  : {res_gepa.initial_eval_score} -> {res_gepa.final_eval_score}  (v{opt_gepa.version})")
else:
    print("RUN_OPTIMIZATION=False → dilewati")

### E5.3 Bandingkan semua kandidat pada dataset yang sama, lalu putuskan manual

In [ ]:
if RUN_OPTIMIZATION:
    rows = {}
    for alias in ("production", "candidate-weighted", "candidate-gepa"):
        uri = f"prompts:/{PROMPT_NAME}@{alias}"
        ver = mlflow.genai.load_prompt(uri, cache_ttl_seconds=0).version
        with mlflow.start_run(run_name=f"final-{alias}"):
            res = mlflow.genai.evaluate(
                data=EVAL_DATA_EXT,
                predict_fn=make_predict_fn(uri),
                scorers=[category_match, brevity_scorer],
            )
        rows[alias] = (ver, res.metrics)
        print(f"{alias:20} v{ver:<3} {res.metrics}")
    print("\nPromosikan pemenang dengan: "
          "mlflow.genai.set_prompt_alias(PROMPT_NAME, alias='production', version=<versi>)")

---
## Workshop Recap

You just walked through the full GenAI governance loop:

| Step | API / Tool | What it governs |
| --- | --- | --- |
| Register & version | `mlflow.genai.register_prompt()` | Prompt wording |
| Promote / rollback | `mlflow.genai.set_prompt_alias()` | Which version is live |
| Model access | AI Gateway endpoint | Provider, credentials, routing |
| Typed agents | Pydantic AI `OpenAIResponsesModel` + gateway `base_url` | Application structure |
| Quality measurement | `mlflow.genai.evaluate()` | Objective prompt comparison |
| Auto-improvement | `mlflow.genai.optimize_prompts()` | Data-driven prompt rewrites |

**Suggested exercises:**
1. Register a v3 prompt manually and A/B evaluate it against `@production`
2. Add a custom `@scorer` for brevity (penalize long responses)
3. Enable usage tracking on your gateway endpoint and inspect traces
4. Swap `MetaPromptOptimizer` for `GepaPromptOptimizer` with a larger dataset

**References:**
- [Prompt Registry docs](https://mlflow.org/docs/latest/genai/prompt-registry/)
- [AI Gateway docs](https://mlflow.org/docs/latest/genai/governance/ai-gateway/)
- [Optimize prompts](https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/)